# 02 — Web of Science reproducibility

This notebook reproduces the **Web of Science aggregation stage** of the Paper-1 methodology from **frozen platform exports**.

## What is and is not reproduced

| Included | Not included |
|----------|----------------|
| Loading `sources/web_of_science/q*/{A,B}/savedrecs.xls` | Live Web of Science API / web search |
| Core RE/TE query selection (79 rows) | Re-running original database searches |
| Platform-level `Article Title` deduplication (62 rows) | API credentials |

Frozen exports are the **reproducibility boundary**. Historical exploratory live-search cells in the original notebook are archival only (`legacy/notebooks/wos.ipynb`, ignored).


## Included queries

**Included (core funnel):** `q1`, `q2`, `q3`, `q4`, `q16`, `q17` (strategies A/B when present).

**Excluded from the core Paper-1 search:** Information Extraction and Generation families  
`q31–q34`, `q46–q49`, `q61–q64`.

Concatenation order is frozen to the historical discovery order that produced the legacy unique-title checkpoint (`legacy/checkpoints/wos_reducido.csv`):  
`q1 → q17 → q2 → q16 → q4 → q3` (strategy **A** then **B**).

This mirrors the Scopus core query selection.


In [ ]:
from pathlib import Path
import sys

# Resolve repository root from this notebook location.
ROOT = Path.cwd()
if not (ROOT / "sources" / "web_of_science").is_dir():
    candidate = Path.cwd().resolve()
    for parent in [candidate, *candidate.parents]:
        if (parent / "sources" / "web_of_science").is_dir() and (parent / "src").is_dir():
            ROOT = parent
            break

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from re_te_lowresources.web_of_science import (
    CORE_QUERY_ORDER,
    EXPECTED_CORE_ROWS,
    EXPECTED_UNIQUE_ROWS,
    TITLE_COLUMN,
    discover_wos_exports,
    reproduce_wos,
)

print("Repository root:", ROOT)
print("Core query order:", CORE_QUERY_ORDER)
print("Title column:", TITLE_COLUMN)


## Loading `.xls` exports

Each frozen export is `savedrecs.xls` (xlrd). Discovery is deterministic and does not depend on filesystem listing order.


In [ ]:
exports = discover_wos_exports(ROOT / "sources" / "web_of_science")
for path in exports:
    print(path.relative_to(ROOT))
print(f"\n{len(exports)} export files")


## Aggregation, checkpoints, and outputs

Shared implementation: `re_te_lowresources.web_of_science.reproduce_wos`.

1. **Core concat** → `data/automatic/web_of_science/wos_core.csv` (expected **79**)
2. **`Article Title` unique, keep first** → `data/automatic/web_of_science/wos_unique.csv` (expected **62**)

`wos_core.csv` is the core Paper-1 search concat, not the historical all-query exploratory aggregate (`legacy/checkpoints/wos.csv`, 97 unique-title rows after all-query dedup).


In [ ]:
result = reproduce_wos(ROOT, validate=True, write=True)

print(f"WoS core records: {result.raw_rows} (expected {EXPECTED_CORE_ROWS})")
print(f"WoS deduplicated records: {result.dedup_rows} (expected {EXPECTED_UNIQUE_ROWS})")
print("Wrote:", result.core_path.relative_to(ROOT))
print("Wrote:", result.unique_path.relative_to(ROOT))

assert result.raw_rows == EXPECTED_CORE_ROWS
assert result.dedup_rows == EXPECTED_UNIQUE_ROWS
print("PASS")


## Compact checkpoint summary


In [ ]:
core = result.core
unique = result.unique

summary = {
    "core_rows": len(core),
    "unique_rows": len(unique),
    "columns": len(core.columns),
    "title_column": TITLE_COLUMN,
    "queries_in_core": list(dict.fromkeys(core["query"].tolist())),
    "rows_per_query_core": core["query"].value_counts().reindex(CORE_QUERY_ORDER).to_dict(),
}
for key, value in summary.items():
    print(f"{key}: {value}")
